In [1]:
from rdflib import Graph
from rdfine import GraphReader
from compilers import (
    PipelineGenerator,
    ProjectBuilder,
    LdioConfigCompiler)

#### Loading the graph

In [2]:
# Loading the graph
input_folder = "..\\data\\"
catalog_graph = Graph()
catalog_graph.parse(input_folder + "catalog.ttl", publicID = "file:///workspace/pipeline/")
catalog_reader = GraphReader(catalog_graph)
catalog_reader = catalog_reader.infer(input_folder + "inference_rules.yaml")

#### Compiling the pipeline build

`PipelineGenerator` orchestrates the full chain. It runs `PipelineExtractor` first (the only compiler that needs the `pipeline_id`, and it seeds the `tcs:PipelineBuild` node), then loops over `Compiler._registry`, invoking every compiler whose `applies_to` trigger becomes true against the growing build graph. Execution order emerges from those triggers rather than from any class-level rank.

In [3]:
pipeline_id = ":DemonstratorPipeline"
gen = PipelineGenerator(pipeline_id, catalog_reader.graph)
build_graph = gen.compile()

# Which compilers actually ran?
[cls.__name__ for cls in gen.compilers]

['PipelineExtractor',
 'PipelineAssembler',
 'LdioConfigCompiler',
 'RdfcConfigCompiler',
 'SemanticWorksCompiler',
 'DockerComposeCompiler']

#### Inspecting the compiled files

Every compiler that produces a file attaches it to the `tcs:PipelineBuild` as a `tcs:File` node via `tcs:compiledFile`. The build graph is now self-describing: it knows which files should be written, where, and with what content.

`ProjectBuilder` collects those nodes into a DataFrame on `builder.files` for inspection before any IO happens.

In [4]:
builder = ProjectBuilder(build_graph)

for _, row in builder.files.iterrows():
    print(f"=== {row['filepath']}/{row['filename']} ===")
    print(row['content'])
    print()

=== ./docker-compose.yml ===
services:
  virtuoso:
    image: redpencil/virtuoso:1.4.0
    environment:
      SPARQL_UPDATE: 'true'
  ldio-pipeline-starter:
    image: curlimages/curl
    volumes:
    - ./ldio_pipeline.yml:/pipeline.yml:ro
    command: 'sh -c " sleep 30 && curl -X POST -H ''content-type: application/yaml''
      http://ldio-workbench:8080/admin/api/v1/pipeline --data-binary @/pipeline.yml
      "'
  error-alert:
    image: lblod/loket-error-alert-service
    volumes:
    - ./config/error-alert/:/config/
    environment:
      DEBUG: false
      EMAIL_FROM: info@dishacled.com
      EMAIL_TO: everyone@world.com
  ldio-workbench:
    container_name: ldio-workbench
    image: ldes/ldi-orchestrator:2.8.0-SNAPSHOT
    ports:
    - 8080:8080
  berichtencentrum-deliver-email-service:
    image: lblod/berichtencentrum-deliver-email-service
    environment:
      EMAIL_CRON_PATTERN: '*/1 * * * * *'
      HOURS_DELIVERING_TIMEOUT: '1'
      MAILFOLDER_URI: http://data.lblod.info/

#### Writing the project to disk

`ProjectBuilder.write(target_dir)` materializes every collected file under the given directory, creating parent folders as needed. Existing files at the same path are overwritten. The call returns the absolute paths it wrote.

In [5]:
written = builder.write("../out/demonstrator")
for path in written:
    print(path)

C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\demonstrator\docker-compose.yml
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\demonstrator\rdfc\pipeline.ttl
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\demonstrator\ldio\config.yml


#### Inspecting compiler internals

`PipelineGenerator` keeps the compiler instances it ran on `gen.compilers`, keyed by class. Each instance retains its intermediate state — useful for debugging when an output doesn't look right.

In [6]:
gen.compilers[LdioConfigCompiler].df_steps

,component,type,name,config
0,ldio:SparqlConstructTransformer,Transformer,Ldio:SparqlConstructTransformer,:config_2
1,ldio:RdfAdapter,Adapter,Ldio:RdfAdapter,NaN
2,ldio:HttpInPoller,Input,Ldio:HttpInPoller,:config_6
3,ldio:HttpOut,Output,Ldio:HttpOut,NaN


#### Inspecting what each compiler added and removed

Every compiler on `gen.compilers` exposes `.added_triples` and `.removed_triples` — `GraphReader` views over the delta between its `input_reader` (snapshot at construction time) and its `output_reader` (final state after `compile()`). Together they make the compilation process fully transparent: for any compiler, you can see exactly which triples it contributed and which it removed.

`SemanticWorksCompiler` is a good example because it does both: it strips the old `tcs:literal` (or `tcs:embedded`) body of each SemanticWorks `tcs:DockerComposeConfig` and writes back an updated one with the step's config folded into the service `environment`.

In [6]:
from compilers import SemanticWorksCompiler

sw = gen.compilers[SemanticWorksCompiler]

print(f"Triples added by SemanticWorksCompiler: {len(sw.added_triples.df)}")
print(f"Triples removed by SemanticWorksCompiler: {len(sw.removed_triples.df)}")

print("\n--- added ---")
display(sw.added_triples.df)
print("\n--- removed ---")
display(sw.removed_triples.df)

Triples added by SemanticWorksCompiler: 2
Triples removed by SemanticWorksCompiler: 2

--- added ---


,sub,pred,obj,sub_type,obj_type
0,:config_9,tcs:literal,"{""services"": {""error-alert"": {""image"": ""lblod/...",<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>
1,:config_6,tcs:literal,"{""services"": {""berichtencentrum-deliver-email-...",<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>



--- removed ---


,sub,pred,obj,sub_type,obj_type
0,:config_9,tcs:literal,\r\nerror-alert:\r\n image: lblod/loket-err...,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>
1,:config_6,tcs:literal,\r\n berichtencentrum-deliver-email-service:\...,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>
